In [ ]:
!pip install -q torch torchvision

In [ ]:
# This copies the entire dataset from your Drive to the Colab hardware
!cp -r /content/drive/MyDrive/satellite_image/model_dataset/EuroSat /content/EuroSat_Local

In [ ]:
# =====================================================================
# MOUNT GOOGLE DRIVE (Only for saving the models)
# =====================================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# =====================================================================
# 1. HYPERPARAMETERS & CONFIGURATION
# =====================================================================
BATCH_SIZE = 256
NUM_CLASSES = 10
MAX_EPOCHS = 30
LEARNING_RATE = 1e-3
PATIENCE = 5

# --- CHANGED: Now pointing to local Colab storage for instant loading ---
BASE_DATA_DIR = '/content/EuroSat_Local'
TRAIN_DIR = os.path.join(BASE_DATA_DIR, 'train')
VAL_DIR = os.path.join(BASE_DATA_DIR, 'val')

# --- SAVE DIRECTORY (Still saving securely to your Drive) ---
DRIVE_SAVE_DIR = '/content/drive/MyDrive/satellite_image/EuroSAT_Models'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(DRIVE_SAVE_DIR, "dinov2_eurosat_best.pth")
CHECKPOINT_PATH = os.path.join(DRIVE_SAVE_DIR, "dinov2_eurosat_latest_checkpoint.pth")
RESUME_FROM_CHECKPOINT = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# =====================================================================
# 2. DATA LOADERS
# =====================================================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=(0, 180)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(f"Loading training data from {TRAIN_DIR}...")
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)

print(f"Loading validation data from {VAL_DIR}...")
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=val_transforms)

CLASSES = train_dataset.classes

# Set to num_workers=2 because local NVMe storage can handle it perfectly
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# =====================================================================
# 3. MODEL, OPTIMIZER (ADAM), & EARLY STOPPING
# =====================================================================
class DINOv2LinearProbe(nn.Module):
    def __init__(self, model_name='dinov2_vitb14', num_classes=10):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', model_name)
        for param in self.backbone.parameters():
            param.requires_grad = False

        embed_dim = self.backbone.embed_dim
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(embed_dim),
            nn.Dropout(0.2),
            nn.Linear(embed_dim, num_classes)
        )

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        return self.classifier(features)

class EarlyStopping:
    def __init__(self, patience=5, save_path="best_model.pth"):
        self.patience = patience
        self.save_path = save_path
        self.best_loss = float('inf')
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), self.save_path)
            print(f"   ✓ Validation loss improved ({val_loss:.4f}). Saved BEST model.")
        else:
            self.counter += 1
            print(f"   ! No loss improvement. EarlyStopping Counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

model = DINOv2LinearProbe().to(DEVICE)
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')
early_stopper = EarlyStopping(patience=PATIENCE, save_path=BEST_MODEL_PATH)

# =====================================================================
# 4. CHECKPOINT RESUME LOGIC
# =====================================================================
start_epoch = 0

if RESUME_FROM_CHECKPOINT and os.path.exists(CHECKPOINT_PATH):
    print(f"\n[INFO] Loading checkpoint '{CHECKPOINT_PATH}'...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    early_stopper.best_loss = checkpoint['early_stopper_best_loss']
    early_stopper.counter = checkpoint['early_stopper_counter']

    print(f"[INFO] Successfully resumed! Starting from Epoch {start_epoch+1}")
elif RESUME_FROM_CHECKPOINT:
    print(f"[WARNING] Checkpoint file '{CHECKPOINT_PATH}' not found. Starting from scratch.")

# =====================================================================
# 5. TRAINING LOOP WITH STATE SAVING
# =====================================================================
print("\nStarting Training...")

try:
    for epoch in range(start_epoch, MAX_EPOCHS):
        model.train()
        total_loss, correct, total_samples = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data).item()
            total_samples += labels.size(0)

        train_acc = (correct / total_samples) * 100
        avg_train_loss = total_loss / total_samples

        # Validation Pass
        model.eval()
        val_correct, val_total, val_loss_total = 0, 0, 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)

                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss_total += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels.data).item()
                val_total += labels.size(0)

        val_acc = (val_correct / val_total) * 100
        avg_val_loss = val_loss_total / val_total

        print(f"Epoch [{epoch+1:02d}/{MAX_EPOCHS}] | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Val Loss: {avg_val_loss:.4f}")

        # Save Latest Checkpoint for fail-safe/transfer
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'early_stopper_best_loss': early_stopper.best_loss,
            'early_stopper_counter': early_stopper.counter
        }
        torch.save(checkpoint, CHECKPOINT_PATH)

        # Check Early Stopping
        early_stopper(avg_val_loss, model)
        if early_stopper.early_stop:
            print(f"\n[INFO] Early stopping triggered after {PATIENCE} stagnant epochs.")
            break

except KeyboardInterrupt:
    print("\n[!] Training manually interrupted by user.")
except Exception as e:
    print(f"\n[!] Training interrupted due to error: {e}")

finally:
    print("\n--- Model Checkpoint Verification ---")
    if os.path.exists(BEST_MODEL_PATH):
        model.load_state_dict(torch.load(BEST_MODEL_PATH))
        print(f"✓ Rescued and loaded the BEST model weights from '{BEST_MODEL_PATH}'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Loading training data from /content/EuroSat_Local/train...
Loading validation data from /content/EuroSat_Local/val...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main



Starting Training...
Epoch [01/30] | Train Acc: 82.12% | Val Acc: 85.69% | Val Loss: 0.4265
   ✓ Validation loss improved (0.4265). Saved BEST model.
Epoch [02/30] | Train Acc: 92.94% | Val Acc: 89.43% | Val Loss: 0.3184
   ✓ Validation loss improved (0.3184). Saved BEST model.
Epoch [03/30] | Train Acc: 94.12% | Val Acc: 90.12% | Val Loss: 0.2847
   ✓ Validation loss improved (0.2847). Saved BEST model.
Epoch [04/30] | Train Acc: 94.61% | Val Acc: 90.52% | Val Loss: 0.2658
   ✓ Validation loss improved (0.2658). Saved BEST model.
Epoch [05/30] | Train Acc: 95.03% | Val Acc: 92.15% | Val Loss: 0.2323
   ✓ Validation loss improved (0.2323). Saved BEST model.
Epoch [06/30] | Train Acc: 95.28% | Val Acc: 93.19% | Val Loss: 0.2077
   ✓ Validation loss improved (0.2077). Saved BEST model.
Epoch [07/30] | Train Acc: 95.27% | Val Acc: 92.85% | Val Loss: 0.2120
   ! No loss improvement. EarlyStopping Counter: 1/5
Epoch [08/30] | Train Acc: 95.55% | Val Acc: 93.39% | Val Loss: 0.1937
   ✓ Vali